In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
pip install ipywidgets

In [3]:
import os, ast, re, warnings
import pandas as pd
import torch

warnings.filterwarnings("ignore")

from transformers import T5Tokenizer, T5ForConditionalGeneration

CSV_PATH   = "/content/drive/MyDrive/spells_master.csv"
MODEL_NAME = "t5-base"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head(3)

(1333, 20)


,name,desc,higher_levels,level,school,classes,subclasses,cast_time,range,duration,verbal,somatic,material,material_desc,concentration,ritual,damage_type,dc_type,attack_type,source
0,Prismatic Wall,"A shimmering, multicolored plane of light form...",NaN,9,Abjuration,['Wizard'],[],1 action,60 feet,10 minutes,True,True,False,NaN,False,False,NaN,NaN,NaN,wotc-srd
1,Symbol,"When you cast this spell, you inscribe a harmf...",NaN,7,Abjuration,"['Bard', 'Cleric', 'Wizard']",[],1 minute,Touch,Until dispelled or triggered,True,True,True,"Mercury, phosphorus, and powdered diamond and ...",False,False,NaN,NaN,NaN,wotc-srd
2,Teleport,This spell instantly transports you and up to ...,NaN,7,Conjuration,"['Bard', 'Sorcerer', 'Wizard']",[],1 action,10 feet,Instantaneous,True,False,False,NaN,False,False,NaN,NaN,NaN,wotc-srd


In [5]:
# ── Phase 1: data-cleaning helpers ────────────────────────────────────────────

def parse_list_col(x):
    """Safely parse a column that may be a Python list literal, CSV string, or NaN."""
    if pd.isna(x) or x == "": return []
    if isinstance(x, list):   return x
    try:    return ast.literal_eval(x)
    except: return [s.strip() for s in str(x).split(",") if s.strip()]

def clean(s):
    """Strip 5e-tools schema tags so targets contain plain English only."""
    if not isinstance(s, str): return ""
    s = re.sub(r"\{@\w+\s([^}]+)\}",        r"\1",              s)  # {@spell Fireball} → Fireball
    s = re.sub(r"\{@\w+\}",                  "",                 s)  # {@hitYourSpellAttack} → ""
    s = re.sub(r"\|[A-Z]{2,}[^|\s]*",        "",                 s)  # |TCE |PHB → ""
    s = re.sub(r"\[&\d+;&\d+\]",             "",                 s)
    s = re.sub(r"\|bestiary[^*\n]*",          "",                 s)
    s = re.sub(r"\|[^|}\s]{1,30}",           "",                 s)
    s = re.sub(r"Vision and Light\|[^\s]*",  "heavily obscured", s)
    s = re.sub(r"difficult terrain\|[^\s]*", "difficult terrain", s)
    return re.sub(r"\s+", " ", s).strip()

print("helpers ready — parse_list_col + clean")

helpers ready — parse_list_col + clean


In [6]:
torch.cuda.empty_cache()
import gc
gc.collect()

222

In [7]:
import torch
torch.cuda.empty_cache()

print(round(torch.cuda.memory_allocated() / 1e9, 2), 'GB used')
print(round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB total')

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
# bfloat16: native on L4/A100, same exponent range as float32 → no NaN risk
model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True
)
model = model.to(device)

print('model loaded')
print(round(torch.cuda.memory_allocated() / 1e9, 2), 'GB used after load')


0.0 GB used
42.41 GB total


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model loaded
0.58 GB used after load


In [8]:
prompt = "describe spell: Fireball"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=40
    )

result = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print(result)

spell: Fireball. Fireball spell: Fireball spell: Fireball spell: Fireball spell: Fireball spell: Fireball spell: Fireball spell: Fireball spell: Fireball


In [9]:
# ── Phase 1: clean every column properly ──────────────────────────────────────

STR_COLS  = ["desc", "higher_levels", "material_desc", "damage_type",
             "dc_type", "attack_type", "school", "cast_time", "range",
             "duration", "source"]
BOOL_COLS = ["verbal", "somatic", "material", "concentration", "ritual"]

for col in STR_COLS:
    if col in df.columns:
        df[col] = df[col].fillna("").apply(clean)

for col in BOOL_COLS:
    if col in df.columns:
        df[col] = df[col].apply(lambda x:
            bool(x) if isinstance(x, bool)
            else str(x).strip().lower() in ("true", "yes", "1", "t"))

for col in ["classes", "subclasses"]:
    if col in df.columns:
        df[col] = df[col].apply(parse_list_col)

df["name"]  = df["name"].fillna("").apply(clean)
df["level"] = pd.to_numeric(df["level"], errors="coerce").fillna(0).astype(int)

# drop blank names and descriptions that are too short to be useful targets
df = df[df["name"].str.strip() != ""].reset_index(drop=True)
df = df[df["desc"].str.len()   >  40].reset_index(drop=True)

# cast_times longer than 40 chars are data noise (e.g. long ritual descriptions)
df["cast_time"] = df["cast_time"].apply(lambda x: x if len(x) < 40 else "")

print(f"rows after cleaning : {len(df)}")
print(f"schools : {sorted(df['school'].unique().tolist())}")

rows after cleaning : 1333
schools : ['Abjuration', 'Conjuration', 'Divination', 'Enchantment', 'Evocation', 'Illusion', 'Necromancy', 'Transmutation']


In [10]:
class SpellDataset(torch.utils.data.Dataset):
    """Tokenizes all examples once at construction time.
    __getitem__ does zero CPU work — tensors are ready to go straight to GPU."""

    def __init__(self, data):
        inputs = tokenizer(
            [item["input"]  for item in data],
            max_length=128,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        targets = tokenizer(
            [item["target"] for item in data],
            max_length=256,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        label_ids = targets["input_ids"].clone()
        label_ids[label_ids == tokenizer.pad_token_id] = -100  # ignore padding in loss

        self.input_ids      = inputs["input_ids"]
        self.attention_mask = inputs["attention_mask"]
        self.labels         = label_ids

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
        }


In [11]:
def build_spell_prompt(row):

    fields = []

    fields.append(f"level: {row['level']}")

    if "school" in row and pd.notna(row["school"]):
        fields.append(f"school: {row['school']}")

    if "cast_time" in row and pd.notna(row["cast_time"]):
        fields.append(f"cast_time: {row['cast_time']}")

    if "range" in row and pd.notna(row["range"]):
        fields.append(f"range: {row['range']}")

    if "duration" in row and pd.notna(row["duration"]):
        fields.append(f"duration: {row['duration']}")

    if "damage_type" in row and pd.notna(row["damage_type"]):
        fields.append(f"damage: {row['damage_type']}")

    return " | ".join(fields)

In [12]:
multi_task_examples = []

for _, row in df.iterrows():

    name = str(row["name"])
    desc = str(row["desc"])

    attrs = build_spell_prompt(row)

    multi_task_examples.append({
        "input": f"describe spell: {name}",
        "target": desc,
        "task": "name_to_desc"
    })

    multi_task_examples.append({
        "input": f"generate name: {desc}",
        "target": name,
        "task": "desc_to_name"
    })

    multi_task_examples.append({
        "input": f"generate description: {attrs}",
        "target": desc,
        "task": "attr_to_desc"
    })

    if pd.notna(row.get("higher_levels")) and str(row["higher_levels"]).strip():
        multi_task_examples.append({
            "input":  f"upcast: {name} | {attrs}",
            "target": str(row["higher_levels"]),
            "task":   "upcast"
        })

    if pd.notna(row.get("school")) and str(row["school"]).strip():
        multi_task_examples.append({
            "input":  f"school of magic: {desc}",
            "target": str(row["school"]),
            "task":   "pred_school"
        })

print(len(multi_task_examples))

pd.DataFrame(multi_task_examples).head()

5849


,input,target,task
0,describe spell: Prismatic Wall,"A shimmering, multicolored plane of light form...",name_to_desc
1,"generate name: A shimmering, multicolored plan...",Prismatic Wall,desc_to_name
2,generate description: level: 9 | school: Abjur...,"A shimmering, multicolored plane of light form...",attr_to_desc
3,"school of magic: A shimmering, multicolored pl...",Abjuration,pred_school
4,describe spell: Symbol,"When you cast this spell, you inscribe a harmf...",name_to_desc


In [13]:
import gc

# ── GPU health-check before probing ─────────────────────────────────────────
_used  = torch.cuda.memory_allocated() / 1e9
_total = torch.cuda.get_device_properties(0).total_memory / 1e9
_free  = _total - torch.cuda.memory_reserved() / 1e9
print(f"GPU before probe: {_used:.2f} GB allocated | {_free:.2f} GB unreserved | {_total:.2f} GB total")
if _used > _total * 0.6:
    print("⚠  WARNING: >60% GPU already allocated — restart runtime for accurate probing!")

def find_max_batch_size(model, tokenizer, device, min_bs=4, max_bs=512):
    """
    Upward-probe for the largest batch size that fits in GPU memory.

    Strategy: start at min_bs and DOUBLE each step until OOM.
    Then binary-search the [last_good, first_bad) interval.

    Why this is better than starting from a high value:
      The allocator only sees SUCCESSFUL allocations during the growth phase.
      OOM-induced fragmentation never poisons the binary-search phase.
    """
    dummy_in  = "describe spell: Fireball"
    dummy_out = "A bright streak of light flashes to a point and explodes in a roar of flame."

    def probe(bs):
        torch.cuda.empty_cache(); gc.collect()
        try:
            enc = tokenizer(
                [dummy_in] * bs,
                max_length=128, truncation=True,
                padding="max_length", return_tensors="pt",
            ).to(device)
            tgt = tokenizer(
                [dummy_out] * bs,
                max_length=256, truncation=True,
                padding="max_length", return_tensors="pt",
            )
            labels = tgt["input_ids"].clone().to(device)
            labels[labels == tokenizer.pad_token_id] = -100

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                outputs = model(**enc, labels=labels)
                f    = outputs.logits.float()
                loss = (f.logsumexp(dim=-1) - f.mean(dim=-1)).mean() * 0 + outputs.loss

            loss.backward()
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache(); gc.collect()
            print(f"  bs={bs:4d}  ✓")
            return True
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            print(f"  bs={bs:4d}  ✗ OOM")
            return False

    # Phase 1: geometric growth — find bracket [lo, hi) around the true max
    lo, hi = min_bs, min_bs
    while hi <= max_bs and probe(hi):
        lo = hi
        hi = min(hi * 2, max_bs + 1)

    if lo == min_bs and not probe(min_bs):
        print("  Even min_bs OOMs — GPU is out of memory!")
        return min_bs

    # Phase 2: binary search inside [lo+1, hi-1] — no large-OOM risk
    best = lo
    low, high = lo + 1, hi - 1
    while low <= high:
        mid = (low + high) // 2
        if probe(mid):
            best = mid; low  = mid + 1
        else:
            high = mid - 1

    torch.cuda.empty_cache(); gc.collect()
    return best


print("probing max batch size (upward-first to avoid fragmentation)…")
_max_bs = find_max_batch_size(model, tokenizer, device, min_bs=4, max_bs=512)

TRAIN_BATCH_SIZE = max(4, int(_max_bs * 0.85))
VAL_BATCH_SIZE   = _max_bs

print(f"\nmax that fits : {_max_bs}")
print(f"train batch   : {TRAIN_BATCH_SIZE}  (85 % of max)")
print(f"val   batch   : {VAL_BATCH_SIZE}")


GPU before probe: 0.59 GB allocated | 41.81 GB unreserved | 42.41 GB total
probing max batch size (upward-first to avoid fragmentation)…
  bs=   4  ✓
  bs=   8  ✓
  bs=  16  ✓
  bs=  32  ✓
  bs=  64  ✓
  bs= 128  ✗ OOM
  bs=  96  ✗ OOM
  bs=  80  ✗ OOM
  bs=  72  ✓
  bs=  76  ✓
  bs=  78  ✗ OOM
  bs=  77  ✓

max that fits : 77
train batch   : 65  (85 % of max)
val   batch   : 77


In [14]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

multi_train, multi_val = train_test_split(
    multi_task_examples,
    test_size=0.1,
    random_state=42
)

# Pre-tokenizes everything once here — __getitem__ is now instant
train_dataset = SpellDataset(multi_train)
val_dataset   = SpellDataset(multi_val)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
)

print(f"train batch size : {TRAIN_BATCH_SIZE}")
print(f"val   batch size : {VAL_BATCH_SIZE}")
print(f"train batches    : {len(train_loader)}")
print(f"val   batches    : {len(val_loader)}")

train batch size : 65
val   batch size : 77
train batches    : 81
val   batches    : 8


In [15]:
import math
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import get_cosine_schedule_with_warmup

EPOCHS     = 25
LR         = 1e-5
SAVE_DIR = "/content/drive/MyDrive/t5_spells_best"  

os.makedirs(SAVE_DIR, exist_ok=True)
checkpoint_path = os.path.join(SAVE_DIR, "training_state.pt")

model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=0.01,
    fused=torch.cuda.is_available(),
)

total_steps = len(train_loader) * EPOCHS
scheduler   = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps,
)

start_epoch        = 0
best_val_loss      = float("inf")
train_history      = []
val_history        = []
perplexity_history = []

print(f"ready — {MODEL_NAME} | lr={LR} | label_smoothing=0.1 | epochs={EPOCHS}")


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

ready — t5-base | lr=1e-05 | label_smoothing=0.1 | epochs=25


In [ ]:
import os, math
import torch.nn.functional as F
from tqdm.auto import tqdm

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

VAL_EVERY = 3   # ← run val / probes / checkpoint only every N epochs

PROBE_PROMPTS = [
    ("describe spell: Fireball",        "name→desc"),
    ("describe spell: Misty Step",      "name→desc"),
    ("school of magic: You hurl a mote of fire at a creature within range.",
     "pred_school"),
    ("generate name: Spectral chains restrain enemies and drain their life force.",
     "desc→name"),
    ("generate description: level: 3 | school: Necromancy | damage: Poison | duration: 1 minute",
     "attr→desc"),
]

def sample_generate(prompt, max_new_tokens=100):
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt",
                    truncation=True, max_length=128).to(device)
    with torch.no_grad():
        ids = model.generate(**enc, max_new_tokens=max_new_tokens, num_beams=4)
    return tokenizer.decode(ids[0], skip_special_tokens=True)

def unwrap(m):
    return m._orig_mod if hasattr(m, "_orig_mod") else m


for epoch in range(start_epoch, EPOCHS):
    do_val = (epoch + 1) % VAL_EVERY == 0 or (epoch + 1) == EPOCHS

    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    total_train_loss = 0.0

    train_bar = tqdm(train_loader,
                     desc=f"Ep {epoch+1:02d}/{EPOCHS} [train]",
                     leave=False, dynamic_ncols=True)

    for batch in train_bar:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = F.cross_entropy(
                outputs.logits.float().view(-1, outputs.logits.size(-1)),
                labels.view(-1),
                ignore_index=-100,
                label_smoothing=0.1,
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)
    train_history.append(avg_train_loss)

    print(f"\nEpoch {epoch+1}/{EPOCHS}  Train Loss: {avg_train_loss:.4f}", end="")

    if not do_val:
        print("  (val skipped)")
        print("─" * 55)
        continue

    # ── Validation (every VAL_EVERY epochs) ───────────────────────────────────
    model.eval()
    total_val_loss = 0.0

    val_bar = tqdm(val_loader,
                   desc=f"Ep {epoch+1:02d}/{EPOCHS} [val]  ",
                   leave=False, dynamic_ncols=True)

    with torch.no_grad():
        for batch in val_bar:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )
                step_loss = F.cross_entropy(
                    outputs.logits.float().view(-1, outputs.logits.size(-1)),
                    labels.view(-1),
                    ignore_index=-100,
                    label_smoothing=0.1,
                )
                total_val_loss += step_loss.item()
            val_bar.set_postfix(loss=f"{step_loss.item():.4f}")

    avg_val_loss = total_val_loss / len(val_loader)
    val_history.append(avg_val_loss)
    perplexity = math.exp(avg_val_loss)
    perplexity_history.append(perplexity)

    print(f"\n  Val Loss   : {avg_val_loss:.4f}")
    print(f"  Perplexity : {perplexity:.2f}")

    # ── Sample generations ────────────────────────────────────────────────────
    print("\n  📝 Sample outputs:")
    for prompt, task in PROBE_PROMPTS:
        out = sample_generate(prompt)
        short_prompt = prompt[:55] + ("…" if len(prompt) > 55 else "")
        print(f"  [{task}]")
        print(f"    IN : {short_prompt}")
        print(f"    OUT: {out[:150]}")

    # ── Checkpoint ────────────────────────────────────────────────────────────
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        raw = unwrap(model)
        raw.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        torch.save(
            {
                "epoch"             : epoch,
                "optimizer_state"   : optimizer.state_dict(),
                "scheduler_state"   : scheduler.state_dict(),
                "best_val_loss"     : best_val_loss,
                "train_history"     : train_history,
                "val_history"       : val_history,
                "perplexity_history": perplexity_history,
            },
            checkpoint_path,
        )
        print(f"\n  ✓ saved best (val loss {best_val_loss:.4f})")

    print("\n" + "─" * 55)
    torch.cuda.empty_cache()


Ep 01/25 [train]:   0%|          | 0/81 [00:00<?, ?it/s]


Epoch 1/25  Train Loss: 11.1444  (val skipped)
───────────────────────────────────────────────────────


Ep 02/25 [train]:   0%|          | 0/81 [00:00<?, ?it/s]


Epoch 2/25  Train Loss: 10.8767  (val skipped)
───────────────────────────────────────────────────────


Ep 03/25 [train]:   0%|          | 0/81 [00:00<?, ?it/s]

...

Ep 12/25 [train]:   0%|          | 0/81 [00:00<?, ?it/s]


Epoch 12/25  Train Loss: 4.8928

Ep 12/25 [val]  :   0%|          | 0/8 [00:00<?, ?it/s]


  Val Loss   : 4.6907
  Perplexity : 108.93

  📝 Sample outputs:
  [name→desc]
    IN : describe spell: Fireball
    OUT: Describe spell: Fireball Fireball Fireball Fireball Fireball Fireball Fireball Fireball Fireball Fireball Fireball Fireball Fireball Fireball Fireball
  [name→desc]
    IN : describe spell: Misty Step
    OUT: Describe spell: Misty Step: Misty Step: Misty Step: Misty Step: Misty Step: Misty Step: Misty Step: Misty Step: Misty Step: Misty Step: Misty Step: Mi
  [pred_school]
    IN : school of magic: You hurl a mote of fire at a creature …
    OUT: You hurl a mote of fire at a creature within range.
  [desc→name]
    IN : generate name: Spectral chains restrain enemies and dra…
    OUT: Spectral chains restrain enemies and drain their life force.
  [attr→desc]
    IN : generate description: level: 3 | school: Necromancy | d…
    OUT: Necromancy level 3 | school: Necromancy | damage: Poison | duration: 1 minute | damage: Poison | duration: 1 minute | damage: Poison |

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
epochs_range = range(1, len(train_history) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Training Summary", fontsize=14, fontweight="bold")

# ── Train Loss ────────────────────────────────────────────────
axes[0].plot(epochs_range, train_history, color="steelblue", linewidth=2, marker="o", markersize=3)
axes[0].set_title("Train Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

# ── Val Loss ──────────────────────────────────────────────────
axes[1].plot(epochs_range, val_history, color="darkorange", linewidth=2, marker="o", markersize=3)
best_epoch = val_history.index(min(val_history)) + 1
axes[1].axvline(best_epoch, color="red", linestyle="--", alpha=0.6, label=f"best epoch {best_epoch}")
axes[1].set_title("Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# ── Perplexity ────────────────────────────────────────────────
axes[2].plot(epochs_range, perplexity_history, color="seagreen", linewidth=2, marker="o", markersize=3)
axes[2].set_title("Validation Perplexity")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Perplexity")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"best val loss  : {min(val_history):.4f}  (epoch {best_epoch})")
print(f"best perplexity: {min(perplexity_history):.2f}")

In [ ]:
def generate(prompt, max_new_tokens=120):

    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(device)

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

In [ ]:
probe_prompts = [

    "describe spell: Fireball",

    "describe spell: Misty Step",

    (
        "generate description: "
        "level: 3 | school: Necromancy | "
        "damage: Poison | duration: 1 minute"
    ),

    (
        "generate description: "
        "level: 7 | school: Conjuration | "
        "range: 500 feet | duration: Instantaneous"
    ),

    (
        "generate description: "
        "level: 2 | school: Illusion | "
        "duration: 10 minutes"
    ),

    (
        "generate description: "
        "level: 0 | school: Evocation | "
        "damage: Lightning | range: 60 feet"
    ),

    (
        "generate description: "
        "level: 6 | school: Transmutation | "
        "duration: 24 hours"
    ),

    (
        "generate name: "
        "A wave of freezing wind blasts outward from you, "
        "dealing cold damage to creatures in a cone."
    ),

    (
        "generate name: "
        "You summon spectral chains that restrain enemies "
        "and drain their life force."
    ),

    (
        "describe spell: Ashen Nova | "
        "school: Evocation | level: 5"
    ),

]

In [ ]:
for i, prompt in enumerate(probe_prompts):


    print(f"Prompt {i + 1}")

    print(prompt)

    print("Output:\n")

    result = generate(
        prompt,
        max_new_tokens=200
    )

    print(result)

    print("\n")